# Lab 1: Computer Vision - Bird's Eye View (BEV) và Lọc Nhiễu

Chào mừng bạn đến với Lab 1. Trong bài thực hành này, chúng ta sẽ tìm hiểu cách xử lý hình ảnh từ camera để tính toán khoảng cách thực tế của vật cản.

---

## Tổng quan

Khi phát triển hệ thống nhận diện cho xe tự hành, camera góc rộng thường gặp hiện tượng "ảo ảnh viễn thị" (Perspective) — các vật ở xa trông sẽ nhỏ hơn và các đường thẳng song song có xu hướng tụ lại một điểm. Điều này khiến việc sử dụng trực tiếp pixel để đo khoảng cách trở nên không chính xác.

Để giải quyết vấn đề này, chúng ta sử dụng phép biến đổi **Inverse Perspective Mapping (IPM)** để chuyển ảnh từ góc nhìn xiên sang góc nhìn từ trên xuống (Bird’s Eye View - BEV).

Sau đó, khoảng cách sẽ được làm mượt bằng bộ lọc **Exponential Moving Average (EMA)** để giảm nhiễu.

---

## Mục tiêu học tập

Sau khi hoàn thành bài thực hành này, bạn sẽ nắm được:

- Cách thiết lập ma trận biến đổi phối cảnh (Homography)
- Cách chuyển đổi tọa độ từ ảnh sang BEV
- Cách hiệu chuẩn pixel → cm
- Cách cài đặt bộ lọc EMA

---

## Bài toán

**Mục tiêu**: Xác định khoảng cách thực tế từ camera đến vật cản.

**Input**: Bounding Box `(x1, y1, x2, y2)`  
**Output**: Khoảng cách (cm) đã làm mượt

In [ ]:
import cv2
import numpy as np

# 1. IPM (Inverse Perspective Mapping)

## Lý thuyết

IPM sử dụng ma trận Homography (3x3) để biến đổi tọa độ:

[x', y', w'] = H * [u, v, 1]

Sau đó chuẩn hóa:
x = x'/w'
y = y'/w'

Ta tạo ma trận này bằng cách ánh xạ 4 điểm:
- SRC (ảnh gốc)
- DST (BEV)

In [ ]:
def create_perspective_matrix(width, height):
    src_ratios = ((0.35, 0.45), (0.25, 0.90), (0.75, 0.90), (0.65, 0.45))
    dst_ratios = ((0.25, 0.00), (0.25, 1.00), (0.75, 1.00), (0.75, 0.00))

    src = np.float32([(x * width, y * height)   for x, y in src_ratios])
    dst = np.float32([(x * width, y * height)   for x, y in dst_ratios])

    M = cv2.getPerspectiveTransform(src, dst)
    return M

## Biến đổi tọa độ điểm

Ta sử dụng điểm **bottom-center của bounding box** để đại diện cho vị trí vật thể trên mặt đất.

In [ ]:
def transform_point(cx, cy, M):
    point = np.array([[[float(cx), float(cy)]]], dtype=np.float32)
    
    point_bev = cv2.perspectiveTransform(point, M)
    
    return point_bev[0][0]

# 2. Calibration (Pixel → cm)

Sau khi có tọa độ BEV, ta tính khoảng cách theo trục y:

distance = (frame_height - y_bev) * cm_per_pixel

Trong đó:
- cm_per_pixel được đo thực nghiệm

In [ ]:
def calculate_real_distance(y_bev, frame_height):
    
    distance_pixel = frame_height - y_bev

    if distance_pixel <= 0:
        return 9999.0

    cm_per_pixel = 30.0 / 472.0
    return distance_pixel * cm_per_pixel

# 3. EMA Filter

EMA giúp làm mượt tín hiệu:

S_t = alpha * X_t + (1 - alpha) * S_(t-1)

In [ ]:
def apply_ema_filter(raw, prev, alpha=0.4):
    if prev is None:
        return raw
    
    smoothed = alpha * raw + (1 - alpha) * prev 
    return smoothed

# 4. Test toàn bộ pipeline

In [ ]:
# Giả lập bounding box
x1, y1, x2, y2 = 100, 120, 200, 300

# Tính bottom-center
cx = (x1 + x2) / 2
cy = y2

# Tạo ma trận
M = create_perspective_matrix(640, 480)

# Transform
x_bev, y_bev = transform_point(cx, cy, M)

# Distance
raw_dist = calculate_real_distance(y_bev, 480)

# EMA
filtered = apply_ema_filter(raw_dist, None)

print("Raw distance:", raw_dist)
print("Filtered distance:", filtered)